# Data Cleaning

**Purpose:** Combine the three booking files, anonymize customer information, address data quality issues, extract useful features, and save a cleaned dataset.

**Input:** `2024.csv`, `2025.csv`, `2026.csv`

**Output:** `booking_clean.csv`

In [21]:
import sys
sys.path.append("..")

import pandas as pd
from pathlib import Path
from src.anonymize import anonymize 

RAW = Path("../data/raw/")
OUT = Path("../data/processed/bookings_clean.csv")

FILES = ["2024.csv", "2025.csv", "2026.csv"]

log = []

## 1. Load & Combine

In [50]:
# Check if the file exits
for f in FILES:
    print(f, (RAW/f).exists())


2024.csv True
2025.csv True
2026.csv True


In [51]:
# Load data & combine
data = {}
for f in FILES:
    data[f] = pd.read_csv(RAW/f)
    print(f, data[f].shape)


# Check if the columns match
base = set(data["2024.csv"].columns)
for f in FILES:
    print(f, "matches 2024:", set(data[f].columns) == base)

# Stack the files together + create source column
frames = [] # create a list to hold the dataframes temporarily before combining
for f in FILES:
    df = data[f].copy()
    df["source"] = f
    frames.append(df)

df = pd.concat(frames, ignore_index=True)
print(df.shape)
log.append(("Combined 3 files", None, len(df)))
for col in df.columns:
    print(col)


2024.csv (6690, 11)
2025.csv (21720, 11)
2026.csv (21053, 11)
2024.csv matches 2024: True
2025.csv matches 2024: True
2026.csv matches 2024: True
(49463, 12)
date
created_by
number
phone_number
status
client_name
service_name
staff_name
service_price
start_time
end_time
source


## 2. Anonymize
The raw datefiles contain the name and phone number of customers, which should be anonymized with a new identifier.

In [52]:
# Rename to match the anynomization function
df = df.rename(columns={
    "phone_number": "phone",
    "client_name": "name"
})

df = anonymize(df, id_source="phone", drop_cols = ("name","phone"))

print("Unique customers:", df["id"].nunique())
print("Rows without customer_id:", df["id"].isna().sum())
print("name/phone still present:", {"name", "phone"} & set(df.columns))
log.append(("Anonymized customers", None, len(df)))



Unique customers: 5219
Rows without customer_id: 0
name/phone still present: set()


## 3. Clean

In [53]:
# Profile dataset
# Profile dataset
profile = pd.DataFrame({
    "non_null": df.count(),
    "missing": df.isna().sum(),
    "unique": df.nunique(),
    "dtype": df.dtypes
})

profile = profile.sort_values("missing", ascending=False)
profile

,non_null,missing,unique,dtype
service_name,49462,1,57,str
staff_name,49462,1,25,str
service_price,49462,1,28,float64
start_time,49462,1,50,str
end_time,49462,1,50,str
date,49463,0,801,str
created_by,49463,0,2,str
number,49463,0,38927,int64
status,49463,0,3,str
source,49463,0,3,str


In [ ]:
# Missing values
print("Missing values by column:")
print(df.isna().sum())
print("Rows with missing values:", df.isna().any(axis=1).sum())
before = len(df)
df = df.dropna()
print(f"Dropped {before - len(df)} rows; {len(df)} remain")
log.append(("Dropped rows with missing service/time/price", before - len(df), len(df)))


Missing values by column:
date             0
created_by       0
number           0
status           0
service_name     1
staff_name       1
service_price    1
start_time       1
end_time         1
source           0
id               0
start            1
end              1
date_parsed      0
dtype: int64
Rows with missing values: 1
Rows with missing values: 1
Dropped 1 rows; 49462 remain


In [54]:
# Checking duplicates
print("Rows:", len(df))
print("Exact duplicate rows:", df.duplicated().sum())
df_noid = df.drop(columns="id")
print("Duplicates (without id):",df_noid.duplicated().sum())
log.append(("Checked duplicates", None, len(df)))

Rows: 49463
Exact duplicate rows: 0
Duplicates (without id): 0


In [ ]:
# Data types parsing
    # Check datetime columns
print(df[["date", "start_time", "end_time"]].sample(5, random_state=1))

df["start"] = pd.to_datetime(df["date"] + " " + df["start_time"], errors="coerce")
df["end"]   = pd.to_datetime(df["date"] + " " + df["end_time"],   errors="coerce")
df["date_parsed"] = pd.to_datetime(df["date"], errors="coerce")

    # Numeric columns
df['service_price'] = pd.to_numeric(df['service_price'], errors='coerce')


             date start_time end_time
44386  2026-03-31      10:15    11:15
25735  2025-02-25      12:30    13:45
12103  2025-10-01      10:30    11:30
8676   2025-12-03      17:00    18:15
19568  2025-06-14      10:00    10:15


In [ ]:
# Fix spelling errors (HTML code &amp; should be &)
fixes = {
    "SHELLAC MANI &amp; PEDI":     "SHELLAC MANI & PEDI",
    "KID MANICURE &amp; PEDICURE": "KID MANICURE & PEDICURE",
    "TINTING &amp; EYEBROW WAX":   "TINTING & EYEBROW WAX",
    "CHIN &amp; SIDEBURNS WAX":    "CHIN & SIDEBURNS WAX",
}

n = df["service_name"].isin(fixes).sum()
df["service_name"] = df["service_name"].replace(fixes)
print(f"Fixed {n} rows")
print(df["service_name"].str.contains("&amp;", na=False).sum())
log.append(("Fixed spelling errors in service_name", n, len(df)))

Fixed 0 rows
0


## 4. Feature Engineering

In [ ]:
# Year, month, day of the week, hour, is_weekend
df["year"] = df["date_parsed"].dt.year
df["month"] = df["date_parsed"].dt.month
df["day_of_week"] = df["date_parsed"].dt.day_name()
df["hour"] = df["start"].dt.hour
df['is_weekend'] = df["day_of_week"].isin(["Saturday", "Sunday"])

# Duration in minutes
df['duration'] = (df['end'] - df['start']).dt.total_seconds() / 60

# Days since last booking
df = df.sort_values(by=["id", "start"])
df['days_since_last_booking'] = df.groupby('id')['start'].diff().dt.days

# Type of service
print(df["service_name"].value_counts(dropna=False))
SERVICE_GROUPS = {
    "Add-ons": [
        "NAIL ART", "AIR BRUSH DESIGN", "MEDIUM OR LONG LENGTH",
        "CAT EYES", "CHROME", "SHAPE CHANGE",
    ],
    "Natural Nails": [
        "MANICURE", "PEDICURE", "SHELLAC MANICURE", "SHELLAC PEDICURE",
        "MANI & PEDI", "SHELLAC MANI & PEDI",
        "SHELLAC POLISH CHANGE (HANDS)", "SHELLAC POLISH CHANGE (TOES)",
        "DELUXE PEDICURE (REGULAR)", "DELUXE PEDICURE (SHELLAC)",
        "DELUXE PEDICURE + MANICURE SHELLAC",
    ],
    "Kids Services": [
        "KID HAND POLISH", "KID TOE POLISH", "KID MANICURE",
        "KID PEDICURE", "KID MANICURE & PEDICURE",
    ],
    "Bare Necessities": [
        "HAND POLISH", "TOE POLISH", "NAIL REPAIR", "CUT DOWN",
        "ACRYLIC TAKE OFF", "SHELLAC TAKE OFF",
        "HAND POLISH WITH FRENCH", "TOE POLISH WITH FRENCH",
    ],
    "Waxing Services": [
        "EYEBROWS WAX", "UPPER LIP WAX", "CHIN WAX", "CHIN & SIDEBURNS WAX",
        "FULL FACE WAX", "UNDER ARMS WAX", "HALF ARMS WAX", "FULL ARMS WAX",
        "HAND OR FEET WAX", "TUMMY WAX", "CHEST WAX", "BACK WAX",
        "BIKINI LINE WAX", "BRAZILIAN WAX", "HALF LEG WAX", "FULL LEG WAX",
        "TINTING & EYEBROW WAX",
    ],
    "Nails Enhancement": [
        "BIO GEL REFILL", "BIO GEL NEW SET", "ACRYLIC REFILL", "ACRYLIC NEW SET",
        "UV GEL REFILL", "UV GEL NEW SET", "SOLAR REFILL", "SOLAR NEW SET",
    ],
}
# Flip to {service: group}. Uppercase the keys so capitalization can't cause a miss.
SERVICE_TO_TYPE = {s.upper(): g for g, items in SERVICE_GROUPS.items() for s in items}

df["service_type"] = df["service_name"].str.strip().str.upper().map(SERVICE_TO_TYPE)
unmapped = df.loc[df["service_type"].isna(), "service_name"].value_counts()
print("Unmapped:", len(unmapped), "services,", unmapped.sum(), "rows")
print(unmapped)

assert df["service_type"].notna().all(), "Some services are unmapped"
print(df["service_type"].value_counts())
log.append(("Feature Engineering", None, len(df)))


service_name
BIO GEL REFILL                        10939
PEDICURE                               6608
SHELLAC PEDICURE                       5198
ACRYLIC REFILL                         4570
SHELLAC MANICURE                       4249
ACRYLIC NEW SET                        3177
NAIL ART                               1800
SHELLAC MANI & PEDI                    1662
SHELLAC POLISH CHANGE (HANDS)          1207
BIO GEL NEW SET                        1181
UV GEL REFILL                          1036
MANI & PEDI                             928
MANICURE                                800
ACRYLIC TAKE OFF                        725
EYEBROWS WAX                            712
SHELLAC POLISH CHANGE (TOES)            536
MEDIUM OR LONG LENGTH                   529
NAIL REPAIR                             517
SOLAR REFILL                            389
SHELLAC TAKE OFF                        361
UV GEL NEW SET                          242
Deluxe Pedicure (Regular)               241
Deluxe Pedicure (Sh

## 5. Validation & Save